In [ ]:
%pip install opencv-python-headless

In [ ]:
%pip install tqdm

In [ ]:
!pip install albumentations

In [ ]:
import cv2
import os
import random
import numpy as np
import albumentations as A
import matplotlib.pyplot as plt

In [ ]:
import cv2
import os
import random
import numpy as np
import albumentations as A
import matplotlib.pyplot as plt

class BDD100KAugmentor:
    def __init__(self, output_base_dir, use_stage1=True, use_stage2=True, stage1_p=0.8, stage2_p=0.8):
       
        self.output_img_dir = os.path.join(output_base_dir, 'images')
        self.output_lab_dir = os.path.join(output_base_dir, 'labels')
        os.makedirs(self.output_img_dir, exist_ok=True)
        os.makedirs(self.output_lab_dir, exist_ok=True)

        self.use_stage1 = use_stage1
        self.use_stage2 = use_stage2

        if self.use_stage1:
            self.stage1_transform = A.Compose([
                A.HorizontalFlip(p=0.75),
                A.RandomScale(scale_limit=0.3, p=0.7),
            ], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels']), p=stage1_p)

        if self.use_stage2:
            self.stage2_transform = A.Compose([
                #hue_shift_limit=20 (色相偏移) val_shift_limit=20 (明度偏移) sat_shift_limit=30 (饱和度偏移)：
                A.HueSaturationValue(hue_shift_limit=20, sat_shift_limit=40, val_shift_limit=30, p=0.7),
                # 模拟曝光与逆光：RandomBrightnessContrast contrast_limit=0.2 (对比度限制)
                A.RandomBrightnessContrast(brightness_limit=0.25, contrast_limit=0.25, p=0.7),
                #GaussNoise 模拟传感器在极端环境下的电信号噪点。
                A.GaussNoise(std_range=(0.04, 0.2), p=0.3)
            ], p=stage2_p)

    def _read_yolo_labels(self, label_path):
        bboxes = []
        class_labels = []
        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                for line in f.readlines():
                    data = line.strip().split()
                    if len(data) == 5: 
                        c = int(data[0])
                        x_c, y_c, w, h = [float(x) for x in data[1:]]

                        # --- 新增的修复逻辑：边界强制裁剪 ---
                        # 1. 先把 YOLO 格式转换为边缘坐标
                        x_min = x_c - w / 2.0
                        y_min = y_c - h / 2.0
                        x_max = x_c + w / 2.0
                        y_max = y_c + h / 2.0

                        # 2. 强制把越界的坐标拉回到 0.0 ~ 1.0 之间
                        x_min = max(0.0, min(1.0, x_min))
                        y_min = max(0.0, min(1.0, y_min))
                        x_max = max(0.0, min(1.0, x_max))
                        y_max = max(0.0, min(1.0, y_max))

                        # 3. 重新计算合规的 YOLO 格式 (x_center, y_center, width, height)
                        w_new = x_max - x_min
                        h_new = y_max - y_min
                        x_c_new = x_min + w_new / 2.0
                        y_c_new = y_min + h_new / 2.0

                        # 4. 过滤掉因为裁剪导致宽或高变成 0 的无效框
                        if w_new > 0 and h_new > 0:
                            class_labels.append(c)
                            bboxes.append([x_c_new, y_c_new, w_new, h_new])
                            
        return bboxes, class_labels

    def process(self, image_path, label_path, show_result=False, show_original=False):
        # 1. 加载数据
        image = cv2.imread(image_path)
        if image is None:
            print(f"Warning: Could not read image {image_path}")
            return
            
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        bboxes, class_labels = self._read_yolo_labels(label_path)

        if not bboxes:
            print(f"Warning: No valid bboxes found in {label_path}. Skipping.")
            return

        # 展示原图 (如果需要)
        if show_original:
            self._visualize(image, bboxes, class_labels, title="Original Image")

        # 准备初始变量（防止所有阶段都被跳过时无数据可用）
        img_aug = image.copy()
        bboxes_aug = bboxes.copy()

        # 2. 执行第一阶段：几何变换
        if self.use_stage1:
            augmented = self.stage1_transform(image=img_aug, bboxes=bboxes_aug, class_labels=class_labels)
            img_aug, bboxes_aug = augmented['image'], augmented['bboxes']

        # 3. 执行第二阶段：光照变换 (只改变图，不改变框)
        if self.use_stage2:
            augmented_final = self.stage2_transform(image=img_aug)
            img_aug = augmented_final['image']

        # 4. 保存结果
        base_name = os.path.basename(image_path).split('.')[0]
        new_name = f"{base_name}_aug_{random.randint(1000, 9999)}"
        
        save_img_path = os.path.join(self.output_img_dir, f"{new_name}.jpg")
        cv2.imwrite(save_img_path, cv2.cvtColor(img_aug, cv2.COLOR_RGB2BGR))
        
        save_lab_path = os.path.join(self.output_lab_dir, f"{new_name}.txt")
        with open(save_lab_path, 'w') as f:
            for i in range(len(bboxes_aug)):
                line = f"{class_labels[i]} " + " ".join([f"{x:.6f}" for x in bboxes_aug[i]]) + "\n"
                f.write(line)

        # 5. 展示增强后的结果
        if show_result:
            self._visualize(img_aug, bboxes_aug, class_labels, title="Augmented Image")

    def _visualize(self, image, bboxes, class_labels, title="Preview"):
        h, w, _ = image.shape
        vis_image = image.copy()
        for i, box in enumerate(bboxes):
            x_c, y_c, b_w, b_h = box
            x1 = int((x_c - b_w/2) * w)
            y1 = int((y_c - b_h/2) * h)
            x2 = int((x_c + b_w/2) * w)
            y2 = int((y_c + b_h/2) * h)
            
            # 绘制绿色边框
            cv2.rectangle(vis_image, (x1, y1), (x2, y2), (0, 255, 0), 2)
            # 绘制标签文本（带黑色描边，防止在白底看不清）
            text = str(class_labels[i])
            cv2.putText(vis_image, text, (x1, max(0, y1-10)), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 0), 4)
            cv2.putText(vis_image, text, (x1, max(0, y1-10)), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
        
        plt.figure(figsize=(8, 5))
        plt.imshow(vis_image)
        plt.title(title)
        plt.axis('off')
        plt.show()

    

In [ ]:
augmentor = BDD100KAugmentor(
    output_base_dir='bdd100k_augmented/train',
    use_stage1=True,   
    use_stage2=True,   
    stage1_p=0.9,      
    stage2_p=0.9       
)

augmentor.process(
    image_path='/home/sagemaker-user/InfinifyX/data_preprocessing/bdd100k/train/images/6b31ce2a-d3d78b71.jpg',
    label_path='/home/sagemaker-user/InfinifyX/data_preprocessing/bdd100k/train/labels/6b31ce2a-d3d78b71.txt',
    show_original=True, 
    show_result=True    
)

In [ ]:
import os
import random
try:
    from tqdm import tqdm
except ImportError:
    print("建议安装 tqdm 以显示进度条: !pip install tqdm")
    # 如果没有 tqdm，就用一个假的占位代替
    def tqdm(iterable, **kwargs): return iterable

def batch_targeted_augmentation(augmentor, img_dir, lab_dir, target_classes, process_ratio=0.15):
    """
    针对包含特定类别（稀有类别）的图片进行定向数据增强。
    
    :param augmentor: 实例化的 BDD100KAugmentor 对象
    :param img_dir: 原图片文件夹路径
    :param lab_dir: 原标签文件夹路径
    :param target_classes: 需要触发增强的类别 ID 列表 (如 [1, 2, 3, 6, 7, 8, 9])
    :param process_ratio: 增强数量占原数据集总量的比例，默认 15% (0.15)
    """
    
    # 1. 获取所有图片并打乱顺序
    print("正在扫描图片目录...")
    image_files = [f for f in os.listdir(img_dir) if f.endswith(('.jpg', '.png'))]
    random.shuffle(image_files) # 打乱顺序，保证随机性
    
    total_images = len(image_files)
    target_aug_count = int(total_images * process_ratio)
    
    print(f"总图片数: {total_images}")
    print(f"目标类别 IDs: {target_classes}")
    print(f"计划增强数量 ({process_ratio * 100}%): {target_aug_count} 张")
    
    augmented_count = 0
    
    # 2. 遍历打乱后的图片
    # 使用 tqdm 包装 image_files 来显示进度条
    pbar = tqdm(image_files, desc="增强进度")
    for img_name in pbar:
        # 如果达到了设定的增强数量，停止循环
        if augmented_count >= target_aug_count:
            pbar.set_description("已达到目标数量")
            break
            
        img_path = os.path.join(img_dir, img_name)
        # 假设图片的后缀是 .jpg，标签是对应的 .txt
        lab_name = os.path.splitext(img_name)[0] + '.txt'
        lab_path = os.path.join(lab_dir, lab_name)
        
        if not os.path.exists(lab_path):
            continue # 跳过没有标签文件的图片
            
        # 3. 快速检查标签文件中是否包含目标稀有类别
        contains_target = False
        with open(lab_path, 'r') as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5: # 确保是正常的 YOLO 格式
                    class_id = int(parts[0])
                    # 如果这行标注的 class_id 在我们需要增强的列表中
                    if class_id in target_classes:
                        contains_target = True
                        break # 只要发现一个目标类别，就没必要继续往下读了，直接跳出读取循环
                        
        # 4. 如果包含目标类别，调用你的 Augmentor 进行处理
        if contains_target:
            augmentor.process(
                image_path=img_path, 
                label_path=lab_path, 
                show_original=False, # ⚠️ 批量处理时必须设为 False，否则会疯狂弹窗卡死
                show_result=False    # ⚠️ 批量处理时必须设为 False
            )
            augmented_count += 1
            # 实时更新进度条后缀信息
            pbar.set_postfix({'已生成': augmented_count})

    print(f"\n✅ 批量定向增强完成！共生成了 {augmented_count} 张新图片和标签。")



In [ ]:

augmentor = BDD100KAugmentor(
    output_base_dir='/home/sagemaker-user/InfinifyX/data_preprocessing/bdd100k_augmented/train',
    use_stage1=True,    
    use_stage2=True,    
    stage1_p=0.9,       
    stage2_p=0.9        
)

# 2. 定义哪些类别触发增强 (person, truck, bus, bike, motor, train, other)
# 排除了 car(0), traffic light(4), traffic sign(5)
target_rare_classes = [1, 2, 3, 6, 7, 8, 9] 

# 3. 原数据集路径
IMAGE_DIR = '/home/sagemaker-user/InfinifyX/data_preprocessing/bdd100k/train/images'
LABEL_DIR = '/home/sagemaker-user/InfinifyX/data_preprocessing/bdd100k/train/labels'

# 4. 开始批量执行 (处理 15% 的数据)
batch_targeted_augmentation(
    augmentor=augmentor,
    img_dir=IMAGE_DIR,
    lab_dir=LABEL_DIR,
    target_classes=target_rare_classes,
    process_ratio=0.15
)

In [ ]:
import cv2
import os
import random
import matplotlib.pyplot as plt

def show_random_yolo_result(base_dir):
    """
    单纯从指定的 YOLO 格式数据集目录中，随机抽取一张图片并可视化其 Bounding Box。
    :param base_dir: 基础目录，目录下必须包含 'images' 和 'labels' 两个子文件夹
    """
    img_dir = os.path.join(base_dir, 'images')
    lab_dir = os.path.join(base_dir, 'labels')
    
    # 1. 检查目录是否存在
    if not os.path.exists(img_dir) or not os.path.exists(lab_dir):
        print(f"❌ 错误: 找不到目录 {img_dir} 或 {lab_dir}")
        return

    # 2. 获取所有图片列表
    image_files = [f for f in os.listdir(img_dir) if f.endswith(('.jpg', '.png'))]
    if not image_files:
        print(f"❌ 错误: 在 {img_dir} 中没有找到任何图片。")
        return

    # 3. 随机抽取一张确实有标签的图片
    while True:
        random_img = random.choice(image_files)
        img_path = os.path.join(img_dir, random_img)
        
        # 拼接对应的 .txt 标签路径
        lab_name = os.path.splitext(random_img)[0] + '.txt'
        lab_path = os.path.join(lab_dir, lab_name)
        
        # 确保标签文件存在且有内容
        if os.path.exists(lab_path) and os.path.getsize(lab_path) > 0:
            break

    # 4. 加载图像
    image = cv2.imread(img_path)
    if image is None:
        print(f"❌ 错误: 无法读取图像 {img_path}")
        return
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    img_h, img_w, _ = image.shape

    # 5. 解析 YOLO 标签并画框
    print(f"🎲 正在展示随机抽取的图片: {random_img}")
    with open(lab_path, 'r') as f:
        for line in f.readlines():
            data = line.strip().split()
            if len(data) >= 5:
                class_id = int(data[0])
                # 读取归一化的 YOLO 坐标
                x_c, y_c, b_w, b_h = [float(x) for x in data[1:5]]
                
                # 转换为像素坐标体系 (x1, y1, x2, y2)
                x1 = int((x_c - b_w / 2.0) * img_w)
                y1 = int((y_c - b_h / 2.0) * img_h)
                x2 = int((x_c + b_w / 2.0) * img_w)
                y2 = int((y_c + b_h / 2.0) * img_h)
                
                # 画绿色框
                cv2.rectangle(image, (x1, y1), (x2, y2), (0, 255, 0), 2)
                
                # 画类别标签 (带黑色描边，防止在复杂背景下看不清)
                text = str(class_id)
                cv2.putText(image, text, (x1, max(0, y1-8)), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 0), 4)
                cv2.putText(image, text, (x1, max(0, y1-8)), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

    # 6. 使用 Matplotlib 展示
    plt.figure(figsize=(12, 7))
    plt.imshow(image)
    plt.title(f"Result View: {random_img}", fontsize=14)
    plt.axis('off')
    plt.show()



In [ ]:

TARGET_DIR = '/home/sagemaker-user/InfinifyX/data_preprocessing/bdd100k_augmented/train'

show_random_yolo_result(TARGET_DIR)

In [ ]:
#valid test
augmentor = BDD100KAugmentor(
    output_base_dir='/home/sagemaker-user/InfinifyX/data_preprocessing/bdd100k_augmented/val',
    use_stage1=True,    
    use_stage2=True,    
    stage1_p=0.9,       
    stage2_p=0.9        
)

# 2. 定义哪些类别触发增强 (person, truck, bus, bike, motor, train, other)
# 排除了 car(0), traffic light(4), traffic sign(5)
target_rare_classes = [1, 2, 3, 6, 7, 8, 9] 

# 3. 原数据集路径
IMAGE_DIR = '/home/sagemaker-user/InfinifyX/data_preprocessing/bdd100k/val/images'
LABEL_DIR = '/home/sagemaker-user/InfinifyX/data_preprocessing/bdd100k/val/labels'

# 4. 开始批量执行 (处理 15% 的数据)
batch_targeted_augmentation(
    augmentor=augmentor,
    img_dir=IMAGE_DIR,
    lab_dir=LABEL_DIR,
    target_classes=target_rare_classes,
    process_ratio=0.15
)

In [ ]:

TARGET_DIR = '/home/sagemaker-user/InfinifyX/data_preprocessing/bdd100k_augmented/val'

show_random_yolo_result(TARGET_DIR)

In [ ]:
import cv2
import os
import random
import matplotlib.pyplot as plt

class CopyPasteAugmentor:
    def __init__(self, img_dir, lab_dir, output_base_dir, rare_classes):
        """
        初始化 Copy-Paste 增强器
        :param img_dir: 原图片库路径
        :param lab_dir: 原标签库路径
        :param output_base_dir: 增强后输出的基准目录
        :param rare_classes: 需要抠图粘贴的稀有类别列表，如 [1, 2, 3, 6, 7, 8, 9]
        """
        self.img_dir = img_dir
        self.lab_dir = lab_dir
        self.rare_classes = rare_classes
        
        self.output_img_dir = os.path.join(output_base_dir, 'images')
        self.output_lab_dir = os.path.join(output_base_dir, 'labels')
        os.makedirs(self.output_img_dir, exist_ok=True)
        os.makedirs(self.output_lab_dir, exist_ok=True)
        
        # 建立素材索引字典
        self.rare_index = self._build_rare_index()

    def _build_rare_index(self):
        """
        扫描整个标签库，建立稀有物体的索引 (不加载图片，极速完成)
        返回格式: {class_id: [{'img_name': '...', 'bbox': [x,y,w,h]}, ...]}
        """
        print("🔍 正在扫描数据集，建立稀有类别素材库索引...")
        index = {c: [] for c in self.rare_classes}
        count = 0
        
        lab_files = [f for f in os.listdir(self.lab_dir) if f.endswith('.txt')]
        for lab_name in lab_files:
            lab_path = os.path.join(self.lab_dir, lab_name)
            img_name = lab_name.replace('.txt', '.jpg') # 假设图片为 jpg
            
            with open(lab_path, 'r') as f:
                for line in f.readlines():
                    data = line.strip().split()
                    if len(data) == 5:
                        c = int(data[0])
                        if c in self.rare_classes:
                            index[c].append({
                                'img_name': img_name,
                                'bbox': [float(x) for x in data[1:]]
                            })
                            count += 1
                            
        print(f"✅ 素材库建立完成！共收集到 {count} 个稀有类别目标。")
        return index

    def _get_random_crop(self, target_h, target_w):
        """
        从素材库中随机选取一个目标，将其图像抠出并计算大小
        """
        # 1. 随机选一个稀有类别，再从该类别里随机选一个目标
        valid_classes = [c for c in self.rare_classes if len(self.rare_index[c]) > 0]
        if not valid_classes:
            return None
            
        chosen_class = random.choice(valid_classes)
        chosen_item = random.choice(self.rare_index[chosen_class])
        
        # 2. 加载源图片并计算像素坐标
        src_img_path = os.path.join(self.img_dir, chosen_item['img_name'])
        src_img = cv2.imread(src_img_path)
        if src_img is None: return None
        
        sh, sw, _ = src_img.shape
        x_c, y_c, w, h = chosen_item['bbox']
        
        x1 = max(0, int((x_c - w/2) * sw))
        y1 = max(0, int((y_c - h/2) * sh))
        x2 = min(sw, int((x_c + w/2) * sw))
        y2 = min(sh, int((y_c + h/2) * sh))
        
        if x2 <= x1 or y2 <= y1: return None
        
        # 3. 抠图
        crop = src_img[y1:y2, x1:x2]
        
        # 4. 随机缩放 (0.7 到 1.3倍之间)，增加尺度多样性
        scale = random.uniform(0.7, 1.3)
        new_w = int((x2 - x1) * scale)
        new_h = int((y2 - y1) * scale)
        if new_w <= 0 or new_h <= 0 or new_w > target_w or new_h > target_h:
            return None # 缩放异常或比目标背景图还大，则放弃
            
        crop = cv2.resize(crop, (new_w, new_h))
        return crop, chosen_class, new_w, new_h

    def process(self, target_img_path, target_lab_path, min_paste=1, max_paste=3, show_result=False):
        """
        对单张目标图片执行 Copy-Paste 增强
        :param min_paste/max_paste: 每次最多/最少往图上贴几个稀有目标
        """
        # 1. 加载目标背景图和原标签
        target_img = cv2.imread(target_img_path)
        if target_img is None: return
        th, tw, _ = target_img.shape
        
        target_bboxes = []
        target_classes = []
        if os.path.exists(target_lab_path):
            with open(target_lab_path, 'r') as f:
                for line in f.readlines():
                    data = line.strip().split()
                    if len(data) == 5:
                        target_classes.append(int(data[0]))
                        target_bboxes.append([float(x) for x in data[1:]])

        # 2. 决定这次要贴多少个目标
        num_to_paste = random.randint(min_paste, max_paste)
        
        for _ in range(num_to_paste):
            crop_data = self._get_random_crop(th, tw)
            if not crop_data: continue
            
            crop_img, class_id, cw, ch = crop_data
            
            # 3. 寻找合理的粘贴位置
            # 为了防止飞到天上，假设 y 坐标（高度）限定在画面的中下部 (0.3 到 0.8)
            y_min_limit = int(th * 0.3)
            y_max_limit = th - ch
            if y_max_limit <= y_min_limit: continue
            
            paste_y1 = random.randint(y_min_limit, y_max_limit)
            paste_x1 = random.randint(0, tw - cw)
            paste_y2 = paste_y1 + ch
            paste_x2 = paste_x1 + cw
            
            # 4. 执行粘贴 (直接覆盖像素)
            target_img[paste_y1:paste_y2, paste_x1:paste_x2] = crop_img
            
            # 5. 计算并追加新的 YOLO 标签
            new_x_c = (paste_x1 + cw / 2.0) / tw
            new_y_c = (paste_y1 + ch / 2.0) / th
            new_w = cw / tw
            new_h = ch / th
            
            target_classes.append(class_id)
            target_bboxes.append([new_x_c, new_y_c, new_w, new_h])

        # 6. 保存结果
        base_name = os.path.basename(target_img_path).split('.')[0]
        new_name = f"{base_name}_copypaste_{random.randint(1000,9999)}"
        
        cv2.imwrite(os.path.join(self.output_img_dir, f"{new_name}.jpg"), target_img)
        
        with open(os.path.join(self.output_lab_dir, f"{new_name}.txt"), 'w') as f:
            for c, box in zip(target_classes, target_bboxes):
                line = f"{c} " + " ".join([f"{x:.6f}" for x in box]) + "\n"
                f.write(line)

        # 7. 可视化
        if show_result:
            self._visualize(target_img, target_bboxes, target_classes)

    def _visualize(self, image, bboxes, class_labels):
        vis_image = image.copy()
        vis_image = cv2.cvtColor(vis_image, cv2.COLOR_BGR2RGB)
        h, w, _ = vis_image.shape
        
        for i, box in enumerate(bboxes):
            x_c, y_c, b_w, b_h = box
            x1, y1 = int((x_c - b_w/2)*w), int((y_c - b_h/2)*h)
            x2, y2 = int((x_c + b_w/2)*w), int((y_c + b_h/2)*h)
            
            # 对我们关注的稀有类（>0 的，排除car等），用显眼的红色标注
            color = (255, 0, 0) if class_labels[i] in self.rare_classes else (0, 255, 0)
            thickness = 3 if class_labels[i] in self.rare_classes else 1
            
            cv2.rectangle(vis_image, (x1, y1), (x2, y2), color, thickness)
            cv2.putText(vis_image, str(class_labels[i]), (x1, max(0, y1-5)), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, thickness)
            
        plt.figure(figsize=(12, 7))
        plt.imshow(vis_image)
        plt.title("Copy-Paste Augmentation Result (Rare in Red)")
        plt.axis('off')
        plt.show()


In [ ]:

# ================= 使用示例 =================

# 1. 设置路径
IMAGE_DIR = '/home/sagemaker-user/InfinifyX/data_preprocessing/bdd100k/train/images'
LABEL_DIR = '/home/sagemaker-user/InfinifyX/data_preprocessing/bdd100k/train/labels'
OUTPUT_DIR = '/home/sagemaker-user/InfinifyX/data_preprocessing/bdd100k_copypaste/'

# 2. 定义稀有类 (如摩托车7，大巴1，卡车2 等)
RARE_CLASSES = [1, 2, 3, 6, 7, 8, 9]

# 3. 初始化增强器 (它会稍微花几秒钟建立索引)
cp_augmentor = CopyPasteAugmentor(
    img_dir=IMAGE_DIR, 
    lab_dir=LABEL_DIR, 
    output_base_dir=OUTPUT_DIR, 
    rare_classes=RARE_CLASSES
)

# 4. 从图库里随便挑一张图作为“背景基底”，试着往上贴 2-4 个稀有目标
test_img_name = random.choice([f for f in os.listdir(IMAGE_DIR) if f.endswith('.jpg')])
test_img_path = os.path.join(IMAGE_DIR, test_img_name)
test_lab_path = os.path.join(LABEL_DIR, test_img_name.replace('.jpg', '.txt'))

cp_augmentor.process(
    target_img_path=test_img_path,
    target_lab_path=test_lab_path,
    min_paste=2,
    max_paste=4,
    show_result=True # 开启预览，看看贴进去的物体
)